In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

DATA_DIR = "/Users/RoseKamau/Downloads/archive (1)/EuroSAT"

classes = os.listdir(DATA_DIR)
print("🗂 Classes:", classes)

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

label_encoder = LabelEncoder()
label_encoder.fit(list(train_generator.class_indices.keys()))

print("\n✅ Data loaded & scaled successfully!")
print(f"Total training images: {train_generator.samples}")
print(f"Total validation images: {val_generator.samples}")


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(train_generator.class_indices), activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

early_stop =EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=[early_stop]
)
print("\n✅ Training Complete!")

🗂 Classes: ['Forest', 'validation.csv', 'River', 'Highway', 'AnnualCrop', 'test.csv', 'SeaLake', 'label_map.json', 'HerbaceousVegetation', 'Industrial', 'Residential', 'PermanentCrop', 'Pasture', 'train.csv']
Found 21600 images belonging to 10 classes.
Found 5400 images belonging to 10 classes.

✅ Data loaded & scaled successfully!
Total training images: 21600
Total validation images: 5400


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 684,490 (2.61 MB)

 Trainable params: 684,490 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

/opt/anaconda3/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 130s 188ms/step - accuracy: 0.2469 - loss: 1.9433 - val_accuracy: 0.5502 - val_loss: 1.2994
Epoch 2/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 123s 182ms/step - accuracy: 0.4606 - loss: 1.3920 - val_accuracy: 0.5794 - val_loss: 1.1871
Epoch 3/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 121s 179ms/step - accuracy: 0.5535 - loss: 1.2224 - val_accuracy: 0.6691 - val_loss: 0.9739
Epoch 4/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 122s 181ms/step - accuracy: 0.6054 - loss: 1.1030 - val_accuracy: 0.6941 - val_loss: 0.8685
Epoch 5/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 125s 186ms/step - accuracy: 0.6511 - loss: 0.9887 - val_accuracy: 0.7439 - val_loss: 0.7770
Epoch 6/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 124s 184ms/step - accuracy: 0.6851 - loss: 0.9035 - val_accuracy: 0.7478 - val_loss: 0.7253
Epoch 7/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 124s 184ms/step - accuracy: 0.7003 - loss: 0.8558 - val_accuracy: 0.7493 - val_loss: 0.7198
Epoch 8/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 125s 185ms/step - accuracy: 0.7101 -